# Lab 3 — Skills that build on skills, and a library that learns (EP14)

Two ideas from the theory video that Labs 1–2 haven't done yet

**Part A — Hierarchical composition (§22.2.3)**
1. Skills declare `requires` with a version (`citation-format@1`)
2. Write a **resolver**: the load order · a missing skill · a version that doesn't match
3. See one sub-skill **shared** by several skills, then run a composed skill for real

**Part B — A library that learns, Voyager-style (§22.4, stage 6)**
4. No skill fits → the agent **writes a new skill as code**, with a test
5. **Verify** it by running the test → if it passes, store it in the library
6. The next question → **reuse the stored skill** instead of writing a new one

The paper records are teaching data (the arXiv ids, titles, authors and years are real); the outputs and the code the model writes come from real runs.

> Source: Haggai Roitman, *The Hitchhiker's Guide to Agentic AI*, §22.2.3, §22.4, §22.5 — CC BY-SA 4.0

## 0. Setup

| You need | Details |
|---|---|
| Python 3.10 or later | or Google Colab, which already has it |
| An OpenRouter API key | set up as in Lab 1 (this lab costs less than 0.1 US cents) |
| Packages | `openai`, `python-dotenv`, `pyyaml` (installed in the next cell) |

How to add the key, and why we chose this model, are in Lab 1 — **Colab:** 🔑 Secrets → `OPENROUTER_API_KEY` · **your own machine:** a `.env` file

In [ ]:
%pip install -q "openai>=1.50" python-dotenv pyyaml

In [ ]:
import json, os, re
from pathlib import Path
import yaml
from openai import OpenAI

def load_key():
    try:  # Google Colab Secrets
        from google.colab import userdata
        return userdata.get("OPENROUTER_API_KEY")
    except ImportError:
        pass
    from dotenv import load_dotenv
    load_dotenv()
    return os.environ["OPENROUTER_API_KEY"]

MODEL = "google/gemma-4-26b-a4b-it"   # why this model and provider: Lab 1
EXTRA = {"reasoning": {"enabled": False},
         "provider": {"order": ["makora"], "allow_fallbacks": False}}
client = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=load_key(), max_retries=5)
COST = {"chat": 0.0}

PAPERS = {  # teaching data: real arXiv ids, titles, authors and years; one-line summaries written for this lab
    "2310.08560": {"title": "MemGPT: Towards LLMs as Operating Systems", "authors": "Packer et al.", "year": 2023,
                   "summary": "the LLM pages facts between a small main context and a large external store, like an operating system's virtual memory"},
    "2304.03442": {"title": "Generative Agents: Interactive Simulacra of Human Behavior", "authors": "Park et al.", "year": 2023,
                   "summary": "agents keep a memory stream of observations, score memories by recency, importance and relevance, and write reflections"},
    "2303.11366": {"title": "Reflexion: Language Agents with Verbal Reinforcement Learning", "authors": "Shinn et al.", "year": 2023,
                   "summary": "after a failed try the agent writes a short lesson to its memory and uses it on the next try"},
    "2504.19413": {"title": "Mem0: Building Production-Ready AI Agents with Scalable Long-Term Memory", "authors": "Chhikara et al.", "year": 2025,
                   "summary": "an LLM extracts facts from each conversation and decides to add, update or delete them in a long-term store"},
    "2502.12110": {"title": "A-MEM: Agentic Memory for LLM Agents", "authors": "Xu et al.", "year": 2025,
                   "summary": "each memory is a linked note with keywords and tags, and new notes update links to old ones"},
    "2305.16291": {"title": "Voyager: An Open-Ended Embodied Agent with Large Language Models", "authors": "Wang et al.", "year": 2023,
                   "summary": "a Minecraft agent that stores verified code as a growing skill library and reuses it"},
    "2210.03629": {"title": "ReAct: Synergizing Reasoning and Acting in Language Models", "authors": "Yao et al.", "year": 2023,
                   "summary": "the model alternates reasoning steps and tool actions in one loop"},
    "2302.04761": {"title": "Toolformer: Language Models Can Teach Themselves to Use Tools", "authors": "Schick et al.", "year": 2023,
                   "summary": "a model learns by itself where to insert API calls in text"},
}

def search_papers(query: str, max_results: int = 5) -> str:
    words = [w for w in re.findall(r"[a-z]+", query.lower()) if len(w) > 2]
    hits = sorted(((sum(w in (p["title"] + " " + p["summary"]).lower() for w in words), pid)
                   for pid, p in PAPERS.items()), reverse=True)
    return json.dumps([{"id": pid, "title": PAPERS[pid]["title"]} for score, pid in hits[:max_results] if score])

def get_paper(paper_id: str) -> str:
    if paper_id not in PAPERS:
        return json.dumps({"error": f"unknown id {paper_id}; call search_papers first"})
    return json.dumps({"id": paper_id, **PAPERS[paper_id]})

FUNCS = {"search_papers": search_papers, "get_paper": get_paper}
SPECS = {
    "search_papers": {"type": "function", "function": {"name": "search_papers",
        "description": "Search the paper library by keywords in the title or summary. Returns ids and titles.",
        "parameters": {"type": "object", "properties": {"query": {"type": "string"},
                       "max_results": {"type": "integer"}}, "required": ["query"]}}},
    "get_paper": {"type": "function", "function": {"name": "get_paper",
        "description": "Get one paper's full record (title, authors, year, summary) by its arXiv id.",
        "parameters": {"type": "object", "properties": {"paper_id": {"type": "string"}}, "required": ["paper_id"]}}},
}

def run_agent(system, task, tools, max_steps=8):
    """The simple loop from Lab 1."""
    msgs = [{"role": "system", "content": system}, {"role": "user", "content": task}]
    for _ in range(max_steps):
        r = client.chat.completions.create(model=MODEL, temperature=0, extra_body=EXTRA, messages=msgs,
                                           tools=[SPECS[t] for t in tools] or None)
        COST["chat"] += r.usage.cost
        msg = r.choices[0].message
        if not msg.tool_calls:
            return msg.content
        msgs.append(msg.model_dump(exclude_none=True))
        for call in msg.tool_calls:
            args = json.loads(call.function.arguments)
            print(f"  🔧 {call.function.name}({args})")
            msgs.append({"role": "tool", "tool_call_id": call.id, "content": FUNCS[call.function.name](**args)})
    return None

def ask_json(system, user, schema, name):
    """One model call with strict structured output (EP02); returns the parsed JSON.
    max_tokens: strict JSON can loop on whitespace until the limit (EP02) — a cut-off answer raises JSONDecodeError."""
    r = client.chat.completions.create(model=MODEL, temperature=0, extra_body=EXTRA, max_tokens=2000, messages=[
            {"role": "system", "content": system}, {"role": "user", "content": user}],
        response_format={"type": "json_schema", "json_schema": {"name": name, "strict": True, "schema": schema}})
    COST["chat"] += r.usage.cost
    return json.loads(r.choices[0].message.content)

BASE = "You are a research assistant for university students."

## A1. Skills that declare the skills they need

```
literature-review 1.2.0 ──▶ paper-summary 1.0.0
                    └────▶ citation-format 1.1.0 ◀── reading-list
weekly-report ──▶ literature-review, meeting-notes (❓ not in the library)
thesis-chapter ──▶ literature-review, citation-format@2 (❓ the library has 1.1.0)
```

`requires: ["citation-format@1"]` = needs citation-format **major version 1** (1.0, 1.1, … all fine, but not 2.0)

In [ ]:
def skill(name, version, requires=(), tools=(), instructions=""):
    return {"name": name, "version": version, "requires": list(requires), "tools": list(tools),
            "instructions": instructions}

REGISTRY = {s["name"]: s for s in [
    skill("paper-summary", "1.0.0", tools=["search_papers", "get_paper"], instructions=
          "Write for first-year students. Call search_papers, then get_paper for every paper you use; use only what "
          "get_paper returns. Each sentence cites one fetched paper id in square brackets, e.g. [2310.08560]."),
    skill("citation-format", "1.1.0", tools=["get_paper"], instructions=
          "Format each reference as: Authors (Year). Title. arXiv:<id>. Use only fields from get_paper."),
    skill("literature-review", "1.2.0", requires=["paper-summary@1", "citation-format@1"], instructions=
          "1. Use the paper-summary skill to write one short paragraph per paper (at most 2 sentences each).\n"
          "2. Add one sentence that compares the papers.\n"
          "3. End with a 'References' list, using the citation-format skill, one line per paper cited."),
    skill("reading-list", "1.0.0", requires=["citation-format@1"], instructions="List papers on a topic as APA references."),
    skill("weekly-report", "1.0.0", requires=["literature-review@1", "meeting-notes@1"]),
    skill("thesis-chapter", "1.0.0", requires=["literature-review@1", "citation-format@2"]),
]}

## A2. The resolver

**Before running**, the orchestrator resolves the whole graph (§22.2.3): sub-skills come before the skills that use them (topological order),
and it stops at once if a skill is missing, a version doesn't match, or the graph loops back (not a DAG)

In [ ]:
def resolve(name, order=None, path=()):
    order = [] if order is None else order
    if name in order:
        return order
    if name in path:
        raise ValueError(f"cycle: {' → '.join(path + (name,))}")
    for dep in REGISTRY[name]["requires"]:
        dep_name, major = dep.split("@")
        if dep_name not in REGISTRY:
            raise LookupError(f"{name} needs {dep_name}, which is not in the library")
        have = REGISTRY[dep_name]["version"]
        if have.split(".")[0] != major:
            raise LookupError(f"{name} needs {dep_name} {major}.x, the library has {have}")
        resolve(dep_name, order, path + (name,))
    order.append(name)
    return order

In [ ]:
for name in ["literature-review", "reading-list", "weekly-report", "thesis-chapter"]:
    try:
        print(f"✅ {name:18} → {resolve(name)}")
    except (LookupError, ValueError) as e:
        print(f"❌ {name:18} → {e}")

In [ ]:
def used_by(sub):
    return [n for n, s in REGISTRY.items() if any(d.split("@")[0] == sub for d in s["requires"])]

print("citation-format is used by:", used_by("citation-format"))

## A3. Run the composed skill

Activate every skill in the resolved order — the sub-skills come first in the system prompt

In [ ]:
order = resolve("literature-review")
system = BASE + "".join(f"\n\n# Skill: {n} {REGISTRY[n]['version']}\n{REGISTRY[n]['instructions']}" for n in order)
tools = sorted({t for n in order for t in REGISTRY[n]["tools"]})
print(run_agent(system, "Write a short literature review of 3 papers on agent memory.", tools))

## B1. A library that learns: writing a new skill as code

Voyager-style: a skill is **verified code**, stored for reuse
Every skill is a function `run(papers)` that takes only the paper library · the model writes `code` + `test` · we run the test before storing it

⚠️ `exec` runs code the model wrote on our machine — do this **only in a lab**; real systems run it in a sandbox (EP08)

In [ ]:
LIBRARY = Path("skills/learned")   # delete this folder to start again from an empty library
LIBRARY.mkdir(parents=True, exist_ok=True)

WRITE = ("You write reusable Python skills for a research assistant. A skill is one function run(papers) that takes "
         "ONLY papers (no other arguments). papers is a dict: arXiv id -> {title, authors, year, summary}. "
         "It returns a JSON-serializable value. Make it general, useful for similar questions later. Standard library only. "
         "Also write test: Python lines with assert statements that call run() on a small papers dict built in the test. "
         "name: short kebab-case. description: one sentence a router can match later.")

In [ ]:
def verify(code, test):
    ns = {}
    try:
        exec(code, ns)
        exec(test, ns)
        return None
    except Exception as e:
        return f"{type(e).__name__}: {e}"

def learn(task, tries=3):
    schema = {"type": "object", "additionalProperties": False, "required": ["name", "description", "code", "test"],
              "properties": {k: {"type": "string"} for k in ["name", "description", "code", "test"]}}
    feedback = ""
    for i in range(1, tries + 1):
        try:
            s = ask_json(WRITE, task + feedback, schema, "skill")
        except json.JSONDecodeError:
            print(f"  ✍️ try {i}: the answer was cut off")
            feedback = "\n\nYour last answer was cut off. Keep the code and test short."
            continue
        error = verify(s["code"], s["test"])
        print(f"  ✍️ try {i}: {s['name']} → {error or 'test passed'}")
        if not error:
            folder = LIBRARY / s["name"]
            folder.mkdir(exist_ok=True)
            (folder / "SKILL.md").write_text(f"---\nname: {s['name']}\ndescription: {s['description']}\n"
                                             f"version: 1.0.0\nverified: true\n---\n")
            (folder / "skill.py").write_text(s["code"])
            return s["name"]
        feedback = f"\n\nYour last code failed its test: {error}. Fix it."
    return None

## B2. Reuse the stored skills

`solve(question)`: the router picks from the library → nothing fits (`none`) → `learn` → run `run(PAPERS)` → the model answers from the code's result

In [ ]:
def pick(question):
    names = [p.parent.name for p in sorted(LIBRARY.glob("*/SKILL.md"))]
    menu = "\n".join(f"- {n}: {yaml.safe_load((LIBRARY / n / 'SKILL.md').read_text().split('---')[1])['description']}"
                     for n in names)
    schema = {"type": "object", "additionalProperties": False, "required": ["skill"],
              "properties": {"skill": {"type": "string", "enum": names + ["none"]}}}
    return ask_json(f"Pick the code skill whose result can answer the question, or 'none'.\n\nSkills:\n{menu}",
                    question, schema, "pick")["skill"]

def solve(question):
    name = pick(question)
    print(f"📚 library: {sorted(p.name for p in LIBRARY.iterdir())} → {name}")
    if name == "none":
        name = learn(question)
    ns = {}
    exec((LIBRARY / name / "skill.py").read_text(), ns)
    result = ns["run"](PAPERS)
    print(f"  ▶ {name}(PAPERS) = {json.dumps(result)[:160]}")
    r = client.chat.completions.create(model=MODEL, temperature=0, extra_body=EXTRA, messages=[
        {"role": "system", "content": "Answer the question from the skill's result only, in one sentence."},
        {"role": "user", "content": f"Question: {question}\nResult of {name}: {json.dumps(result)}"}])
    COST["chat"] += r.usage.cost
    print(f"  💬 {r.choices[0].message.content}")

In [ ]:
for question in ["How many papers per year are in the library?",
                 "How many papers from 2023 are in the library?",
                 "List the papers from newest to oldest.",
                 "Which papers in the library are from the most recent year?"]:
    print(f"\n❓ {question}")
    solve(question)

In [ ]:
for folder in sorted(LIBRARY.iterdir()):
    print(f"# {folder.name}\n{(folder / 'SKILL.md').read_text()}{(folder / 'skill.py').read_text()}\n")
print(f"cost: ${COST['chat']:.5f}")

## Summary

| Part | What we saw |
|---|---|
| A: composition | `requires` + versions → a topological order · stops before running when a skill is missing or a version doesn't match · `citation-format` shared by three |
| A: run | `literature-review` has no tools of its own, yet gets cited paragraphs + an APA list from its sub-skills |
| B: learning | no skill → write code + test → verify → store · the next question reuses it |

**Exercises**
1. Make the graph loop: let `citation-format` require `literature-review@1`, and read the error
2. Add `meeting-notes` 1.0.0 to the library, and resolve `weekly-report` again
3. Part B: write our own test (not the model's) that checks a known answer, e.g. 2023 must have 6 papers, and use it before storing

**Next — Lab 4:** the same skills with a real framework — `deepagents`